Importing the self-made decision tree and preparting the data

In [23]:
import pandas as pd
import numpy as np
from model_functions.Decision_Tree import Decision_Tree


train= pd.read_csv('claims_train.csv')
test = pd.read_csv('claims_test.csv')

#Cleaning weird values that found during cleaning
train = train[train['Exposure'] <= 1].copy()
test =test[test['Exposure']<=1].copy()

# Adding Risk column
train['Risk'] = train['ClaimNb'] / train['Exposure'] 
test['Risk'] =  test['ClaimNb'] / test['Exposure']

#Encoding
train_encoded=pd.get_dummies(train, columns=['VehBrand', 'VehGas', 'Region'], drop_first=True)#Encoding categorical values
area_map={'A':1,'B':2,'C':3,'D':4,'E':5,'F':6}
train_encoded['Area']=train_encoded['Area'].map(area_map)

test_encoded=pd.get_dummies(test, columns=['VehBrand', 'VehGas', 'Region'], drop_first=True)#Encoding categorical values
area_map={'A':1,'B':2,'C':3,'D':4,'E':5,'F':6}
test_encoded['Area']=test_encoded['Area'].map(area_map)



Building the tree

In [6]:
# #Prepare
X_train=train_encoded.drop(columns=['ClaimNb','Exposure', 'IDpol','Risk'])
y_train=train_encoded['Risk']
# #Train
our_tree=Decision_Tree(max_depth=6,min_samples_split=1000)
our_tree.fit(X_train.values,y_train.values)
#Print structure
our_tree.print_tree(our_tree.tree, feature_names=X_train.columns.tolist())

if VehAge <= 0:
  if VehGas_Regular <= False:
    if Region_R54 <= False:
      if VehPower <= 7:
        if VehPower <= 5:
          if VehBrand_B3 <= False:
            Predict Risk = 0.239
          else:
            Predict Risk = 1.259
        else:
          if VehBrand_B12 <= False:
            Predict Risk = 0.451
          else:
            Predict Risk = 1.331
      else:
        if VehPower <= 9:
          if DrivAge <= 25:
            Predict Risk = 0.672
          else:
            Predict Risk = 0.125
        else:
          if VehPower <= 10:
            Predict Risk = 1.654
          else:
            Predict Risk = 0.315
    else:
      Predict Risk = 2.427
  else:
    if VehBrand_B12 <= False:
      if Region_R25 <= False:
        if Density <= 8346:
          if Density <= 323:
            Predict Risk = 0.470
          else:
            Predict Risk = 0.171
        else:
          Predict Risk = 1.124
      else:
        Predict Risk = 3.878
    else:
      if VehPo

Using our tree for prediction

In [7]:
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score
X_test=test_encoded.drop(columns=['ClaimNb','Exposure', 'IDpol','Risk'])
predictions = our_tree.predict(X_test.values)
y_test=test_encoded['Risk']
mse=mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test,predictions)
r2 = r2_score(y_test, predictions)

print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"R²:   {r2:.4f}")

MSE:  26.895058
RMSE: 5.186045
MAE:  0.492785
R²:   0.0088


The model is weak, want to check that the problem is not with my Tree code. Want to compare results with library version of the decision tree.

In [8]:
from sklearn.tree import DecisionTreeRegressor
sk_tree = DecisionTreeRegressor(max_depth=6,min_samples_split=1000,  random_state=42) #same set up
sk_tree.fit(X_train.values, y_train.values)
y_pred_sklearn = sk_tree.predict(X_test.values)

mse = mean_squared_error(y_test, y_pred_sklearn)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_sklearn)
r2 = r2_score(y_test, y_pred_sklearn)

print(f"sklearn DecisionTreeRegressor results:")
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"R²:   {r2:.4f}")

sklearn DecisionTreeRegressor results:
MSE:  26.895058
RMSE: 5.186045
MAE:  0.492785
R²:   0.0088


Ok, the sklearn tree is as bad as ours :D Then the problem is in a set up, need some other approach. Trying to do a classification of presense of Risk to get P(Risk>0) and then Regression on only nonzero-risk cases, then multiply probabilities.

In [9]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# --- Step 1: Classification (Risk > 0 or not) ---
y_class = (y_train > 0).astype(int)
clf = DecisionTreeClassifier(max_depth=6, min_samples_split=1000, random_state=42)
clf.fit(X_train.values, y_class)

# --- Step 2: Regression (only on nonzero-risk cases) ---
mask = y_train > 0
reg = DecisionTreeRegressor(max_depth=6, min_samples_split=500, random_state=42)
reg.fit(X_train.values[mask], y_train.values[mask])

# --- Step 3: Combine predictions ---
p_claim = clf.predict_proba(X_test.values)[:, 1]
expected_risk = reg.predict(X_test.values)
final_pred = p_claim * expected_risk

# --- Step 4: Evaluate combined model ---
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_test, final_pred)
r2 = r2_score(y_test, final_pred)
print(f"MAE: {mae:.4f}, R²: {r2:.4f}")


MAE: 0.4976, R²: 0.0070


This approach didn't help. Will try to add new categoric features based on bins in the exploration part.

In [ ]:
X_train['YoungDriver'] = (X_train['DrivAge'] < 25).astype(int)               #Adding age categories
X_train['MidAgeDriver'] = ((X_train['DrivAge'] >= 25) & (X_train['DrivAge'] < 60)).astype(int)
X_train['SeniorDriver'] = (X_train['DrivAge'] >= 60).astype(int)

X_test['YoungDriver'] = (X_test['DrivAge'] < 25).astype(int)               #Adding age categories
X_test['MidAgeDriver'] = ((X_test['DrivAge'] >= 25) & (X_test['DrivAge'] < 60)).astype(int)
X_test['SeniorDriver'] = (X_test['DrivAge'] >= 60).astype(int)

X_train['HighPower'] = (X_train['VehPower'] >= 7).astype(int)        #Adding High Power category
X_test['HighPower'] = (X_test['VehPower'] >=7).astype(int)

X_train['Brand12'] = (X_train['VehBrand_B12'] == True).astype(int)      #Brand b12 seems to be the most risky
X_test['Brand12'] = (X_test['VehBrand_B12'] == True).astype(int)

risky_regions=['R21','R43','R11']                                     #Top 3 most risky regions
X_train['Risky_region'] = train['Region'].isin(risky_regions).astype(int)
X_test['Risky_region']  = test['Region'].isin(risky_regions).astype(int)


X_train['UrbanArea'] = ((X_train['Area'] == 5) & (X_train['Area'] == 6)).astype(int)       #Urban areas are more risky
X_test['UrbanArea'] = ((X_test['Area'] == 5) & (X_test['Area'] == 6)).astype(int)

X_train['HighBonusMalus'] = (X_train['BonusMalus'] >200).astype(int)     #HighBonusMalus spikes risk
X_test['HighBonusMalus'] = (X_test['BonusMalus'] >200).astype(int)

X_train['MidDensity'] = ((X_train['Density'] >= 10000) & (X_train['Density'] < 15000)).astype(int) #Mid Density areas showed more risk
X_test['MidDensity']  = ((X_test['Density']  >= 10000) & (X_test['Density']  < 15000)).astype(int)



,Area,VehPower,VehAge,DrivAge,BonusMalus,Density,VehBrand_B10,VehBrand_B11,VehBrand_B12,VehBrand_B13,...,Region_R94,YoungDriver,MidAgeDriver,SeniorDriver,HighPower,Brand12,UrbanArea,HighBonusMalus,Risky_region,MidDensity
244,6,6,4,61,50,12374,False,False,True,False,...,False,0,0,1,0,1,0,0,1,1
326,6,4,0,30,85,10008,False,False,True,False,...,False,0,1,0,0,1,0,0,1,1
766,6,7,0,50,50,10961,False,False,True,False,...,False,0,1,0,1,1,0,0,1,1
788,6,4,11,40,76,10156,False,False,False,False,...,False,0,1,0,0,0,0,0,1,1
907,6,7,7,28,100,10156,False,False,False,False,...,False,0,1,0,1,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541503,6,6,0,43,50,10477,False,False,True,False,...,False,0,1,0,0,1,0,0,1,1
541526,6,6,3,50,90,10477,False,False,True,False,...,False,0,1,0,0,1,0,0,1,1
541583,6,4,15,51,50,14368,False,False,False,False,...,False,0,1,0,0,0,0,0,1,1
541775,6,15,3,36,57,12750,False,False,True,False,...,False,0,1,0,1,1,0,0,1,1


New features are added ,let's try the tree again

In [34]:

sk_tree_new = DecisionTreeRegressor(max_depth=6,min_samples_split=1000,  random_state=42) # Changed set up a bit
sk_tree_new.fit(X_train.values, y_train.values)
y_pred_sklearn_new = sk_tree_new.predict(X_test.values)

mse = mean_squared_error(y_test, y_pred_sklearn_new)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_sklearn_new)
r2 = r2_score(y_test, y_pred_sklearn_new)

print(f"sklearn DecisionTreeRegressor results:")
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"R²:   {r2:.4f}")

sklearn DecisionTreeRegressor results:
MSE:  26.900174
RMSE: 5.186538
MAE:  0.494168
R²:   0.0086


Engineered features didn't help either. I make a conclusion that a decision tree on its own is not good enought for such imbalanced data.